# SOLUTION WITH SIMULATION: Part 2, Topic 1, Lab B: Power Analysis for Password Bypass

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *This tutorial will introduce you to breaking devices by determining when a device is performing certain operations. Our target device will be performing a simple password check, and we will demonstrate how to perform a basic power analysis.*

**LEARNING OUTCOMES:**

* How power can be used to determine timing information.
* Plotting multiple iterations while varying input data to find interesting locations.
* Using difference of waveforms to find interesting locations.
* Performing power captures with ChipWhisperer hardware (hardware only)


## Prerequisites

Hold up! Before you continue, check you've done the following tutorials:

* ☑ Jupyter Notebook Intro (you should be OK with plotting & running blocks).
* ☑ SCA101 Intro (you should have an idea of how to get hardware-specific versions running).

## Power Trace Gathering

At this point you've got to insert code to perform the power trace capture. There are two options here:
* Capture from physical device.
* Read from a file.

You get to choose your adventure - see the two notebooks with the same name of this, but called `(SIMULATED)` or `(HARDWARE)` to continue. Inside those notebooks you should get some code to copy into the following section, which will define the capture function.

Be sure you get the `"✔️ OK to continue!"` print once you run the next cell, otherwise things will fail later on!

Choose your setup options here:

In [ ]:
SCOPETYPE="OPENADC"
PLATFORM='CWHUSKY'
CRYPTO_TARGET='NONE'
VERSION='HARDWARE'

In [ ]:
if VERSION == 'HARDWARE':
    %run "Lab 2_1B - Power Analysis for Password Bypass (HARDWARE).ipynb"
elif VERSION == 'SIMULATED':
    %run "Lab 2_1B - Power Analysis for Password Bypass (SIMULATED).ipynb"

In [ ]:
trace_test = cap_pass_trace("h\n")

#Basic sanity check
assert(len(trace_test) == 3000)
print("✔️ OK to continue!")

## Exploration

So what can we do with this? While first off - I'm going to cheat, and tell you that we have a preset password that starts with `h`, and it's 5 characters long. But that's the only hint so far - what can you do? While first off, let's try plotting a comparison of `h` to something else.

If you need a reminder of how to do a plot - see the matplotlib section of the **Jupyter Introduction** notebook.

The following cell shows you how to capture one power trace with `h` sent as a password. From there:

1. Try adding the plotting code and see what it looks like.
2. Send different passwords to the device. We're only going to look at the difference between a password starting with `h` and something else right now.
3. Plot the different waveforms.

In [ ]:
#Example - capture 'h' - end with newline '\n' as serial protocol expects that
trace_h = cap_pass_trace("h\n")

print(trace_h)

# ###################
# START SOLUTION
# ###################
cw.plot(cap_pass_trace("h\n")) * cw.plot(cap_pass_trace("0\n"))
# ###################
# END SOLUTION
# ###################

For reference, the output should look something like this:
<img src="img/spa_password_h_vs_0_overview.png" alt="SPA of Power Analysis" width="450"/>

If you are using the `%matplotlib notebook` magic, you can zoom in at the start. What you want to notice is there is two code paths taken, depending on a correct or incorrect path. Here for example is a correct & incorrect character processed:
<img src="img/spa_password_h_vs_0_zoomed.png" alt="SPA of Power Analysis" width="450"/>

OK interesting -- what's next? Let's plot every possible password character we could send.

Our password implementation only recognizes characters in the list `abcdefghijklmnopqrstuvwxyz0123456789`, so we're going to limit it to those valid characters for now.

Write some code in the following block that implements the following algorithm:

    for CHARACTER in LIST_OF_VALID_CHARACTERS:
        trace = cap_pass_trace(CHARACTER + "\n")
        plot(trace)
        
The above isn't quite valid code - so massage it into place! You also may notice the traces are way too long - you might want to make a more narrow plot that only does the first say 500 samples of the power trace.

---
📝**Plotting Note**

If using `matplotlib` for plotting, you might need to add a `plt.figure()` at the start to make a new figure. Otherwise you might find your plot ends up in the figure above! If you don't see the plots, sometimes a `plt.show()` is needed at the end.

---

In [ ]:
# ###################
# START SOLUTION
# ###################
from tqdm.notebook import tqdm
fig = cw.plot()
for c in tqdm('abcdefghijklmnopqrstuvwxyz0123456789'):
    trace = cap_pass_trace(c + "\n")
    fig *= cw.plot(trace[0:1000])
    
fig

# ###################
# END SOLUTION
# ###################

The end result should be if you zoom in, you'll see there is a location where a single "outlier" trace doesn't follow the path of all the other traces. That is great news, since it means we learn something about the system from power analysis.

<img src="img/spa_password_list_char1.png" alt="SPA of Power Analysis against all inputs" width="450"/>

Using your loop - you can also try modifying the analysis to capture a correct "first" character, and then every other wrong second character. Do you see a difference you might be able to detect?

The pseudo-code would look something like this:

    for CHARACTER in LIST_OF_VALID_CHARACTERS:
        trace = cap_pass_trace("h" + CHARACTER + "\n")
        plot(trace)

Give that a shot in your earlier code-block, and then let's try and automate this attack to understand the data a little better.

## Automating an Attack against One Character

To start with - we're going to automate an attack against a **single** character of the password. Since we don't know the password (let's assume), we'll use a strategy of comparing all possible inputs together.

An easy way to do this might be to use something that we know can't be part of the valid password. As long as it's processed the same way, this will work just fine. So for now, let's use a password as `0x00` (i.e., a null byte). We can compare the null byte to processing something else:

In [ ]:
ref_trace = cap_pass_trace("\x01\n")[0:1000]
other_trace = cap_pass_trace("h\n")[0:1000]

cw.plot(ref_trace) * cw.plot(other_trace)

This will plot a trace with an input of "\x00" - a null password! This is an invalid character, and seems to be processed as any other invalid password.

Let's make this a little more obvious, and plot the difference between a known reference & every other capture. You need to write some code that does something like this:

    ref_trace = cap_pass_trace( "\x00\n")

    for CHARACTER in LIST_OF_VALID_CHARACTERS:
        trace = cap_pass_trace(CHARACTER + "\n")
        plot(trace - ref_trace)

Again, you may need to modify this a little bit such as adding code to make a new `figure()`. Also notice in the above example how I reduced the number of samples.


In [ ]:
# ###################
# START SOLUTION
# ###################


ref_trace = cap_pass_trace("h0p\x01\n")[0:1000]
plot = cw.plot({})
for c in 'abcdefghijklmnopqrstuvwxyz0123456789': 
    trace = cap_pass_trace('h0p' + c + "\n")[0:1000]
    plot *= cw.plot(trace - ref_trace)
plot
# ###################
# END SOLUTION
# ###################


OK great - hopefully you now see one major "difference". It should look something like this:
    
<img src="img/spa_password_diffexample.png" alt="SPA with Difference" width="450"/>
    

What do do now? Let's make this thing automatically detect such a large difference. Some handy stuff to try out is the `np.sum()` and `np.abs()` function.

The first one will get absolute values:

```python
import numpy as np
np.abs([-1, -3, 1, -5, 6])

    Out[]: array([1, 3, 1, 5, 6])
```

The second one will add up all the numbers.

```python
import numpy as np    
np.sum([-1, -3, 1, -5, 6])

    Out[]: -2
```

Using just `np.sum()` means positive and negative differences will cancel each other out - so it's better to do something like `np.sum(np.abs(DIFF))` to get a good number indicating how "close" the match was.


In [ ]:
import numpy as np
np.abs([-1, -3, 1, -5, 6])

In [ ]:
import numpy as np
np.sum([-1, -3, 1, -5, 6])

In [ ]:
np.sum(np.abs([-1, -3, 1, -5, 6]))

Taking your above loop, modify it to print an indicator of how closely this matches your trace. Something like the following should work:

    ref_trace = cap_pass_trace( "\x00\n")

    for CHARACTER in LIST_OF_VALID_CHARACTERS:
        trace = cap_pass_trace(CHARACTER + "\n")
        diff = SUM(ABS(trace - ref_trace))

        print("{:1} diff = {:2}".format(CHARACTER, diff))

In [ ]:
# ###################
# START SOLUTION
# ###################

ref_trace = cap_pass_trace( "h0p\x01\n")


for c in 'abcdefghijklmnopqrstuvwxyz0123456789': 
    trace = cap_pass_trace("h0p" + c + "\n")
    diff = np.sum(np.abs(trace - ref_trace))
    
    print("{:1} diff = {:2}".format(c, diff))
    
# ###################
# END SOLUTION
# ###################

Now the easy part - modify your above code to automatically print the correct password character. This should be done with a comparison of the `diff` variable - based on the printed characters, you should see one that is 'higher' than the others. Set a threshold somewhere reasonable (say I might use `25.0` based on one run).

## Running a Full Attack

Finally - let's finish this off. Rather than attacking a single character, we need to attack each character in sequence.

If you go back to the plotting of differences, you can try using the correct first character & wrong second character. The basic idea is exactly the same as before, but now we loop through 5 times, and just build up the password based on brute-forcing each character.

Another way you could attack this is by running through all the characters and picking the one with the largest difference.

Take a look at the following for the basic pseudo-code:

```python
guessed_pw = "" #Store guessed password so far

do a loop 5 times (max password size):

    ref_trace = capture power trace(guessed_pw + "\x00\n")

    for CHARACTER in LIST_OF_VALID_CHARACTERS:
        trace = capture power trace (guessed_pw + CHARACTER + newline)
        diff = SUM(ABS(trace - ref_trace))

        if diff > THRESHOLD:

            guessed_pwd += c
            print(guessed_pw)

            break
```

In [ ]:
# ###################
# START SOLUTION
# ###################

guessed_pw = ""

for _ in range(0, 5):  
    biggest_diff = 0
    biggest_char = '\x01'
    ref_trace = cap_pass_trace(guessed_pw + "\x01\n")
    
    for c in 'abcdefghijklmnopqrstuvwxyz0123456789': 
        trace = cap_pass_trace(guessed_pw + c + "\n")
        diff = np.sum(np.abs(trace - ref_trace))

        if diff > biggest_diff:
            biggest_diff = diff
            biggest_char = c
            
    guessed_pw += biggest_char
    print(guessed_pw)

# ###################
# END SOLUTION
# ###################

You should get an output that looks like this:

    h
    h0
    h0p
    h0px
    h0px3

If so - 🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳 Congrats - you did it!!!!

If not - check some troubleshooting hints below. If you get really stuck, check the `SOLN` version (there is one for both with hardware and simulated).

## Hands-On: Check Your Own Guess

> **Hardware only.** This section talks to the real target over the serial line, so it
> does nothing unless you ran the notebook with `VERSION = 'HARDWARE'`.

**Before you start:**

* `ipywidgets` must be installed - it's already listed in `jupyter/requirements.txt`,
  so a normal course environment has it. If not: `pip install ipywidgets`, then restart
  the kernel.
* Set `VERSION = 'HARDWARE'` near the top of the notebook and get the
  `✔️ OK to continue!` print from the sanity-check cell before running this section.
* You do **not** need to have finished "Running a Full Attack" first - the box below
  just starts pre-filled with `guessed_pw` as a convenience. Type any guess you like.

The attack above *infers* the password from power differences - but it never actually
**asks the target** whether a guess is right. Let's close that loop and make it
hands-on: you type a guess, the target tells you if it worked.

`cap_pass_trace()` only ever hands back a power trace. The Safe-o-matic 3000 firmware,
though, also *talks*: after it checks a guess it prints either

* `Access granted, Welcome!`  -> you're in, or
* `PASSWORD FAIL`  -> try again

and then freezes until the next reset.

The helper below sends **whatever guess you type**, resets the target, reads that reply
back off the serial line, and tells you whether it unlocked - and it also captures and
plots the power trace for your guess, so you can line up *"did it unlock?"* against the
SPA you just did.

Try it as a game: start from the automated result, change one character, and see the
verdict flip. Or ignore the automated attack entirely and hand-walk the password one
character at a time using the difference plots from earlier.

In [ ]:
import time
import matplotlib.pyplot as plt

# This check drives the real target over serial, so it needs the ChipWhisperer.
_HAVE_HW = ("target" in globals()) and ("scope" in globals())

# The automated attack above recovered this; it is the box's starting value below.
# Edit it to test any other guess.
EXPECTED_PW = globals().get("guessed_pw", "h0px3")


def try_password(guess, show_plot=True):
    """Send `guess` to the target's password check and report whether it unlocks.

    Hardware only. Resets the target, sends the guess over serial, reads the reply,
    and returns True on 'Access granted'. Also captures and plots the guess's power
    trace so you can compare it against the SPA from earlier.
    """
    if not _HAVE_HW:
        raise RuntimeError("try_password() needs the ChipWhisperer connected - "
                           "re-run the notebook with VERSION = 'HARDWARE'.")

    guess = guess.strip()
    trace = cap_pass_trace(guess + "\n")

    # cap_pass_trace() already reset the target and sent the guess; the firmware has
    # now printed its verdict. Drain the serial buffer and look for the banner.
    time.sleep(0.05)
    reply = target.read(timeout=100)
    while target.in_waiting():
        reply += target.read(timeout=50)
    granted = ("Welcome" in reply) or ("Access granted" in reply)

    verdict = "ACCESS GRANTED" if granted else "password fail"
    print("guess = {!r:12}  ->  {}".format(guess, verdict))
    if reply.strip():
        print("   device said:", " | ".join(reply.split()))

    if show_plot:
        plt.figure()
        plt.plot(trace[:1000], "g" if granted else "r")
        plt.title("'{}'  ->  {}".format(guess, verdict))
        plt.xlabel("sample")
        plt.ylabel("power")
        plt.show()

    return granted


if _HAVE_HW:
    try_password(EXPECTED_PW, show_plot=False)   # quick text-only self-check
else:
    print("Hands-on check skipped - no hardware (VERSION != 'HARDWARE').")


In [ ]:
if not _HAVE_HW:
    print("The interactive guess box is hardware only "
          "(re-run with VERSION = 'HARDWARE').")
else:
    import ipywidgets as widgets
    from IPython.display import display

    _guess_box = widgets.Text(
        value=EXPECTED_PW,
        placeholder="type a password guess",
        description="Guess:",
    )
    _try_button = widgets.Button(description="Try it", button_style="primary", icon="unlock")
    _result_out = widgets.Output()

    def _run_guess(_=None):
        _result_out.clear_output(wait=True)
        with _result_out:
            ok = try_password(_guess_box.value)
            print("\n" + ("you're in!" if ok else "keep going - change a character and try again"))

    _try_button.on_click(_run_guess)

    # Static fallback: prints straight to the cell output so the verdict shows on
    # GitHub / nbviewer, where the widget below only renders as a blank placeholder.
    print("Checking the starting guess:")
    try_password(_guess_box.value)

    # Interactive: edit the box and click "Try it" to check as many guesses as you like.
    display(widgets.HBox([_guess_box, _try_button]), _result_out)


### Lock it in

*(Hardware only.)* Once the box above says **ACCESS GRANTED**, run the next cell. It
re-sends whatever is currently in the text box, one more time, as an independent check
and records it as `recovered_pw`. If your guess is wrong the `assert` stops you here -
go back, use the difference plots, and fix the offending character.

In [ ]:
if _HAVE_HW:
    recovered_pw = _guess_box.value.strip()
    assert try_password(recovered_pw, show_plot=False), \
        "{!r} did not unlock the target - go back and adjust your guess.".format(recovered_pw)
    print("hands-on password confirmed:", recovered_pw)
else:
    print("Hands-on confirmation skipped - hardware only.")


## Troubleshooting - Always get 'h'

Some common problems you might run into - first, if you get an output which keeps guessing the first character:

    h
    hh
    hhh
    hhhh
    hhhhh

Check that when you run the `cap_pass_trace` inside the loop (checking the guessed password), are you updating the prefix of the password? For example, the old version of the code (guessing a single character) looked like this:

    trace = cap_pass_trace(c + "\n")

But that is always sending our first character only! So we need to send the "known good password so far". In the example code something like this:
   
    trace = cap_pass_trace(guessed_pw + c + "\n")

Where `guessed_pw` progressively grows with the known good start of the password.

#### Troubleshooting - Always get 'a'

This looks like it's always matching the first character:

    h
    ha
    haa
    haaa
    haaaa

Check that you update the `ref_trace` - if you re-use the original reference trace, you won't be looking at a reference where the first N characters are good, and the remaining characters are bad. An easy way to do this is again using the `guessed_pw` variable and appending a null + newline:

    trace = cap_pass_trace(guessed_pw + "\x00\n")

### Hands-On section

* **`RuntimeError: try_password() needs the ChipWhisperer connected`** - `VERSION` was
  `'SIMULATED'`, or the hardware setup cell was never (re-)run. Set
  `VERSION = 'HARDWARE'` near the top of the notebook and re-run from there.
* **Clicking "Try it" does nothing / box just says "hardware only"** - same cause as
  above; the whole cell short-circuits when `_HAVE_HW` is `False`.
* **`ModuleNotFoundError: No module named 'ipywidgets'`** - `pip install ipywidgets`
  (already in `jupyter/requirements.txt` for a normal course environment), then
  restart the kernel and re-run the cell.
* **The box and button don't render, just a blank area / "Widget Javascript not
  detected"** - usually a stale kernel or browser cache. Restart the kernel
  (Kernel -> Restart), refresh the browser tab, and re-run the cell. On very old
  "classic" Jupyter Notebook (not JupyterLab or Notebook 7+) you may also need
  `jupyter nbextension enable --py widgetsnbextension`.
* **A correct-looking password still says "password fail"** - stale bytes left on the
  serial buffer from a previous run/crash can desync the reply. Re-run the setup cell
  that creates `scope`/`target` (or restart the kernel) and try again.
* **Everything times out / no "device said" line appears** - same checklist as the
  automated attack: target actually flashed with `basic-passwdcheck`, USB cable/port
  solid, and no other process (another Jupyter kernel, a second open notebook) holding
  the Husky.

---
## Optional: Attack Your Own Password

Everything above breaks the *shipped* password. If you have hardware you can swap in
your own and run the whole attack again - this time against something the Python side
genuinely doesn't know.

> **Hardware only.** The `SIMULATED` setup replays fixed recordings of `h0px3` and
> cannot respond to a rebuilt target.

## Where the password lives

It's hardcoded in the **target firmware C source**, not in the notebook or in Python:

`../../../firmware/mcu/basic-passwdcheck/basic-passwdcheck.c` (line 73)

```c
char correct_passwd[] = "h0px3";
```

At runtime it's copied onto the stack in `main()` and compared one byte at a time
against your input (lines 103-107). The early `break` on the first wrong byte is
exactly the leak this lab exploits.

It gets compiled into the MCU image (the SAM4S on the Husky) and is plainly visible as
ASCII in the build outputs - `grep -a h0px3 basic-passwdcheck-CWHUSKY.lss` / `.elf` /
`.bin` all hit. (The `.hex` file writes every byte as two hex digits, so there the password is `6830707833`.)

The notebooks only *reference* the value, they don't define it:

* the attack loop (`for _ in range(0, 5)`) assumes length 5;
* the final `assert guessed_pw == 'h0px3'` hardcodes the expected result;
* `EXPECTED_PW` in the hands-on helper just defaults to whatever the attack recovered.

## Yes, you can change it

The full build-and-flash process is documented step by step, with a tested
troubleshooting log, in its own companion notebook in this folder:
**[`Lab 2_1B - Firmware Change and Flash Walkthrough.ipynb`](./Lab%202_1B%20-%20Firmware%20Change%20and%20Flash%20Walkthrough.ipynb)**.
Open it and follow it top to bottom. Short version:

1. **Edit** `correct_passwd[]` in
   `../../../firmware/mcu/basic-passwdcheck/basic-passwdcheck.c` (line 73), e.g.
   `char correct_passwd[] = "zzq7a";`.
2. **Rebuild:** `make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE SS_VER=SS_VER_2_1 -j` from
   `firmware/mcu/basic-passwdcheck/` (or the `HARDWARE` notebook's firmware-build cell).
3. **Reflash:** re-run the `HARDWARE` notebook's `cw.program_target(...)` cell (or the
   companion notebook's flashing cell / `flash_husky.py`).

### Keep it working with the attack

* **Character set:** the attack only sweeps `abcdefghijklmnopqrstuvwxyz0123456789`.
  Pick your password from those characters or the sweep will never try it (the
  firmware itself accepts anything, but the attack loop won't).
* **Length:** the attack loop runs exactly 5 times and the final cell asserts
  `== 'h0px3'`. If you change the length you must also update the loop count
  (`range(0, N)`), that `assert`, and `EXPECTED_PW`. The firmware compares over
  `sizeof(correct_passwd)` (which includes the terminating NUL), so a longer password
  may also need more samples in the capture window (`scope.adc.samples`).
* Nothing else in the firmware needs to change - the trigger and comparison logic are
  length-agnostic.

Once rebuilt and reflashed, re-run the attack section from the top. `guessed_pw` should
converge on your new password, and the hands-on box above will confirm it.

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2020. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>

In [ ]:
assert guessed_pw == 'h0px3', "Failed to break password"